# Azure LLM Inference Trace — Statistics

Input/Output token length statistics and request arrival rate analysis.

All logic is in reusable functions — call them with different parameters to compare configurations easily.

In [2]:
import csv
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
from IPython.display import display, Markdown

## Functions

In [3]:
def load_trace(csv_path: str, max_requests: int = None) -> pd.DataFrame:
    """
    Load Azure trace CSV into a DataFrame.

    Returns DataFrame with columns:
        timestamp, input_tokens, output_tokens, total_tokens, arrival_sec
    where arrival_sec is relative seconds from the first request.
    """
    rows = []
    with open(csv_path, 'r') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if max_requests is not None and i >= max_requests:
                break
            rows.append({
                'timestamp': datetime.fromisoformat(row['TIMESTAMP']),
                'input_tokens': int(row['ContextTokens']),
                'output_tokens': int(row['GeneratedTokens']),
            })

    df = pd.DataFrame(rows)
    df['total_tokens'] = df['input_tokens'] + df['output_tokens']
    df['arrival_sec'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()

    print(f"Loaded {len(df):,} requests from {Path(csv_path).name}")
    print(f"Time span: {df['timestamp'].iloc[0]} → {df['timestamp'].iloc[-1]}")
    print(f"Duration: {df['arrival_sec'].iloc[-1]:.1f}s ({df['arrival_sec'].iloc[-1]/60:.1f} min)")
    return df

In [4]:
def filter_window(df: pd.DataFrame, start_min: float = 0, end_min: float = None) -> pd.DataFrame:
    """
    Filter trace to a relative time window.

    Args:
        df: DataFrame from load_trace()
        start_min: Start of window in minutes (relative to first request)
        end_min: End of window in minutes (None = end of trace)

    Returns:
        Filtered DataFrame with arrival_sec recalculated from window start.
    """
    first_ts = df['timestamp'].iloc[0]
    t_start = first_ts + timedelta(minutes=start_min)

    if end_min is not None:
        t_end = first_ts + timedelta(minutes=end_min)
        mask = (df['timestamp'] >= t_start) & (df['timestamp'] < t_end)
        label = f"{start_min}–{end_min} min"
    else:
        mask = df['timestamp'] >= t_start
        label = f"{start_min} min–end"

    filtered = df[mask].copy()
    if len(filtered) > 0:
        filtered['arrival_sec'] = (filtered['timestamp'] - filtered['timestamp'].iloc[0]).dt.total_seconds()

    print(f"Window [{label}]: {len(filtered):,} requests")
    return filtered

In [5]:
def _percentile_row(arr, name):
    """Build a statistics row for a numeric array."""
    return {
        "Metric": name,
        "Count": len(arr),
        "Mean": f"{np.mean(arr):.2f}",
        "Std": f"{np.std(arr):.2f}",
        "Min": int(np.min(arr)),
        "P10": f"{np.percentile(arr, 10):.0f}",
        "P25": f"{np.percentile(arr, 25):.0f}",
        "Median": f"{np.median(arr):.0f}",
        "P75": f"{np.percentile(arr, 75):.0f}",
        "P90": f"{np.percentile(arr, 90):.0f}",
        "P99": f"{np.percentile(arr, 99):.0f}",
        "Max": int(np.max(arr)),
    }


def token_stats(df: pd.DataFrame, title: str = None):
    """
    Display input/output/total token length statistics.

    Args:
        df: DataFrame from load_trace() or filter_window()
        title: Optional title override
    """
    if len(df) == 0:
        print("No data.")
        return

    table = pd.DataFrame([
        _percentile_row(df['input_tokens'].values, 'Input tokens'),
        _percentile_row(df['output_tokens'].values, 'Output tokens'),
        _percentile_row(df['total_tokens'].values, 'Total tokens (in+out)'),
    ])

    if title:
        display(Markdown(f"### {title}"))
    else:
        display(Markdown("### Token Length Statistics"))
    display(table.set_index('Metric'))

In [6]:
def arrival_stats(df: pd.DataFrame, title: str = None):
    """
    Display request arrival rate and inter-arrival time statistics.

    Args:
        df: DataFrame from load_trace() or filter_window()
        title: Optional title override
    """
    if len(df) < 2:
        print("Not enough data.")
        return

    arr = df['arrival_sec'].values
    max_sec = int(arr[-1]) + 1

    # Per-second counts
    per_sec = np.zeros(max_sec, dtype=int)
    for t in arr:
        per_sec[min(int(t), max_sec - 1)] += 1

    # Per-minute counts
    max_min = int(arr[-1] / 60) + 1
    per_min = np.zeros(max_min, dtype=int)
    for t in arr:
        per_min[min(int(t / 60), max_min - 1)] += 1

    # Inter-arrival times
    iat_ms = np.diff(arr) * 1000

    rows = [
        _percentile_row(per_sec, 'Arrival rate (req/s)'),
        _percentile_row(per_min, 'Arrival rate (req/min)'),
    ]

    if title:
        display(Markdown(f"### {title}"))
    else:
        display(Markdown("### Arrival Rate Statistics"))
    display(pd.DataFrame(rows).set_index('Metric'))

    if len(iat_ms) > 0:
        display(Markdown("### Inter-Arrival Time"))
        display(pd.DataFrame([_percentile_row(iat_ms, 'Inter-arrival time (ms)')]).set_index('Metric'))

In [7]:
def summary(df: pd.DataFrame, label: str = "Full trace"):
    """
    Display a one-row summary of the trace/window.

    Args:
        df: DataFrame from load_trace() or filter_window()
        label: Description label for the window
    """
    if len(df) < 2:
        print("Not enough data.")
        return

    dur = df['arrival_sec'].iloc[-1]
    s = {
        "Window": label,
        "Requests": f"{len(df):,}",
        "Duration (sec)": f"{dur:.1f}",
        "Duration (min)": f"{dur/60:.1f}",
        "Avg req/s": f"{len(df)/dur:.2f}" if dur > 0 else "N/A",
        "Total input tokens": f"{df['input_tokens'].sum():,}",
        "Total output tokens": f"{df['output_tokens'].sum():,}",
        "Avg input len": f"{df['input_tokens'].mean():.1f}",
        "Avg output len": f"{df['output_tokens'].mean():.1f}",
        "Median input len": f"{df['input_tokens'].median():.0f}",
        "Median output len": f"{df['output_tokens'].median():.0f}",
    }

    display(Markdown(f"### Summary — {label}"))
    display(pd.DataFrame([s]).T.rename(columns={0: 'Value'}))

In [8]:
def compare_windows(df: pd.DataFrame, windows: list):
    """
    Compare statistics across multiple time windows.

    Args:
        df: DataFrame from load_trace()
        windows: List of (start_min, end_min) tuples.
                 end_min=None means end of trace.

    Example:
        compare_windows(df, [(0, 15), (15, 30), (0, None)])
    """
    first_ts = df['timestamp'].iloc[0]
    rows = []

    for w_start, w_end in windows:
        t_start = first_ts + timedelta(minutes=w_start)
        if w_end is not None:
            t_end = first_ts + timedelta(minutes=w_end)
            mask = (df['timestamp'] >= t_start) & (df['timestamp'] < t_end)
            label = f"{w_start}–{w_end} min"
        else:
            mask = df['timestamp'] >= t_start
            label = f"{w_start} min–end"

        w = df[mask]
        if len(w) == 0:
            continue

        w_arr = (w['timestamp'] - w['timestamp'].iloc[0]).dt.total_seconds().values
        w_dur = w_arr[-1] if len(w_arr) > 1 else 1

        rows.append({
            'Window': label,
            'Requests': len(w),
            'Avg req/s': f"{len(w)/w_dur:.2f}" if w_dur > 0 else 'N/A',
            'Avg input': f"{w['input_tokens'].mean():.0f}",
            'Med input': f"{w['input_tokens'].median():.0f}",
            'Avg output': f"{w['output_tokens'].mean():.0f}",
            'Med output': f"{w['output_tokens'].median():.0f}",
            'P90 input': f"{np.percentile(w['input_tokens'], 90):.0f}",
            'P90 output': f"{np.percentile(w['output_tokens'], 90):.0f}",
            'P99 input': f"{np.percentile(w['input_tokens'], 99):.0f}",
            'P99 output': f"{np.percentile(w['output_tokens'], 99):.0f}",
        })

    if rows:
        display(Markdown('### Time Window Comparison'))
        display(pd.DataFrame(rows).set_index('Window'))

---
## Usage Examples

In [9]:
# Load trace
df = load_trace("AzureLLMInferenceTrace_conv.csv")

Loaded 19,366 requests from AzureLLMInferenceTrace_conv.csv
Time span: 2023-11-16 18:15:46.680590 → 2023-11-16 19:14:08.402527
Duration: 3501.7s (58.4 min)


In [10]:
# Full trace statistics
token_stats(df)
arrival_stats(df)
summary(df)

### Token Length Statistics

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Input tokens,19366,1154.70,1108.79,2,207,396,1020,1189,2734,4142,14050
Output tokens,19366,211.13,162.87,7,54,85,129,395,424,601,1000
Total tokens (in+out),19366,1365.82,1102.44,64,368,491,1412,1558,2820,4259,14089


### Arrival Rate Statistics

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Arrival rate (req/s),3502,5.53,2.77,0,2,3,5,7,9,13,16
Arrival rate (req/min),59,328.24,82.14,37,248,270,326,380,440,491,507


### Inter-Arrival Time

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Inter-arrival time (ms),19365,180.83,197.86,0,16,48,118,247,421,890,4314


### Summary — Full trace

,Value
Window,Full trace
Requests,"19,366"
Duration (sec),3501.7
Duration (min),58.4
Avg req/s,5.53
Total input tokens,"22,361,870"
Total output tokens,"4,088,665"
Avg input len,1154.7
Avg output len,211.1
Median input len,1020


In [ ]:
# Specific time window: first 15 minutes
w = filter_window(df, 0, 10)
token_stats(w, title="Token Stats (0–15 min)")
arrival_stats(w, title="Arrival Rate (0–15 min)")
summary(w, label="0–15 min")

Window [0–12 min]: 3,470 requests


### Token Stats (0–15 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Input tokens,3470,1172.82,998.00,2,315,410,1046,1171,2374,4111,7930
Output tokens,3470,257.08,173.48,10,54,91,217,405,432,624,1000
Total tokens (in+out),3470,1429.91,988.37,95,423,518,1452,1568,2440,4208,7979


### Arrival Rate (0–15 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Arrival rate (req/s),720,4.82,2.36,0,2,3,5,6,8,10,12
Arrival rate (req/min),12,289.17,39.96,191,261,267,300,311,328,350,353


### Inter-Arrival Time

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Inter-arrival time (ms),3469,207.55,224.47,0,21,57,141,284,484,971,4314


### Summary — 0–15 min

,Value
Window,0–15 min
Requests,"3,470"
Duration (sec),720.0
Duration (min),12.0
Avg req/s,4.82
Total input tokens,"4,069,697"
Total output tokens,"892,077"
Avg input len,1172.8
Avg output len,257.1
Median input len,1046


In [12]:
# Compare multiple windows side by side
compare_windows(df, [
    (0, 15),
    (15, 30),
    (30, 60),
    (60, 120),
    (0, None),  # full trace
])

### Time Window Comparison

,Requests,Avg req/s,Avg input,Med input,Avg output,Med output,P90 input,P90 output,P99 input,P99 output
Window,,,,,,,,,,
0–15 min,4424,4.92,1173,1047,254,205,2423,430,4109,626
15–30 min,5684,6.32,1298,1014,189,107,4080,418,4130,598
30–60 min,9258,5.44,1058,997,204,126,2326,420,4722,584
0 min–end,19366,5.53,1155,1020,211,129,2734,424,4142,601


In [13]:
# Specific time window: first 10 minutes
w = filter_window(df, 0, 10)
token_stats(w, title="Token Stats (0–10 min)")
arrival_stats(w, title="Arrival Rate (0–10 min)")
summary(w, label="0–10 min")

Window [0–10 min]: 2,867 requests


### Token Stats (0–10 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Input tokens,2867,1146.63,976.79,2,242,408,1041,1165,2221,4106,7930
Output tokens,2867,260.27,171.91,10,55,92,217,405,432,620,1000
Total tokens (in+out),2867,1406.90,969.96,95,414,515,1447,1564,2329,4199,7979


### Arrival Rate (0–10 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Arrival rate (req/s),600,4.78,2.39,0,2,3,5,6,8,10,12
Arrival rate (req/min),10,286.70,43.36,191,254,266,286,318,331,351,353


### Inter-Arrival Time

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Inter-arrival time (ms),2866,209.34,230.90,0,21,57,139,284,492,989,4314


### Summary — 0–10 min

,Value
Window,0–10 min
Requests,"2,867"
Duration (sec),600.0
Duration (min),10.0
Avg req/s,4.78
Total input tokens,"3,287,402"
Total output tokens,"746,194"
Avg input len,1146.6
Avg output len,260.3
Median input len,1041


In [14]:
# Specific time window: first 10 minutes
w = filter_window(df, 0, 3)
token_stats(w, title="Token Stats (0–3 min)")
arrival_stats(w, title="Arrival Rate (0–3 min)")
summary(w, label="0–3 min")

Window [0–3 min]: 785 requests


### Token Stats (0–3 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Input tokens,785,964.48,912.56,2,181,380,999,1111,1314,4090,4107
Output tokens,785,259.24,164.08,12,58,99,217,403,429,594,1000
Total tokens (in+out),785,1223.71,926.91,95,334,454,1398,1514,1689,4158,4292


### Arrival Rate (0–3 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Arrival rate (req/s),180,4.36,2.48,0,1,2,4,6,8,10,11
Arrival rate (req/min),3,261.67,56.39,191,206,228,265,297,316,328,329


### Inter-Arrival Time

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Inter-arrival time (ms),784,229.46,290.88,0,18,56,154,296,530,1104,4314


### Summary — 0–3 min

,Value
Window,0–3 min
Requests,785
Duration (sec),179.9
Duration (min),3.0
Avg req/s,4.36
Total input tokens,"757,116"
Total output tokens,"203,500"
Avg input len,964.5
Avg output len,259.2
Median input len,999


In [15]:
# Specific time window: first 10 minutes
w = filter_window(df, 0, 10)
token_stats(w, title="Token Stats (0–10 min)")
arrival_stats(w, title="Arrival Rate (0–10 min)")
summary(w, label="0–10 min")

Window [0–10 min]: 2,867 requests


### Token Stats (0–10 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Input tokens,2867,1146.63,976.79,2,242,408,1041,1165,2221,4106,7930
Output tokens,2867,260.27,171.91,10,55,92,217,405,432,620,1000
Total tokens (in+out),2867,1406.90,969.96,95,414,515,1447,1564,2329,4199,7979


### Arrival Rate (0–10 min)

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Arrival rate (req/s),600,4.78,2.39,0,2,3,5,6,8,10,12
Arrival rate (req/min),10,286.70,43.36,191,254,266,286,318,331,351,353


### Inter-Arrival Time

,Count,Mean,Std,Min,P10,P25,Median,P75,P90,P99,Max
Metric,,,,,,,,,,,
Inter-arrival time (ms),2866,209.34,230.90,0,21,57,139,284,492,989,4314


### Summary — 0–10 min

,Value
Window,0–10 min
Requests,"2,867"
Duration (sec),600.0
Duration (min),10.0
Avg req/s,4.78
Total input tokens,"3,287,402"
Total output tokens,"746,194"
Avg input len,1146.6
Avg output len,260.3
Median input len,1041
